# Multi-Model Kaggle Backend API
Notebook này load cả 5 mô hình (BERT, BiLSTM, TextCNN, SVM, LightGBM) và chạy FastAPI server.

In [ ]:
!pip install pyngrok fastapi uvicorn nest-asyncio transformers emoji nltk scikit-learn lightgbm torch

In [ ]:
import os
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import emoji
import unicodedata
import joblib
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import nltk
from nltk.tokenize import word_tokenize
from scipy.sparse import hstack, csr_matrix

nltk.download('punkt')
nltk.download('punkt_tab')

## 1. PREPROCESSING FUNCTIONS

In [ ]:
# Preprocessing cho BERT
def preprocess_text_bert(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'http[s]?://\S+', '<url>', text)
    text = re.sub(r'www\.\S+', '<url>', text)
    text = re.sub(r'@\w+', '<username>', text)
    text = re.sub(r'\b\d+\b', '<number>', text)
    text = emoji.demojize(text, delimiters=(' :', ': '))
    text = re.sub(r'[^a-z0-9<>: ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Preprocessing cho Machine Learning (SVM, LightGBM)
def preprocess_text_ml(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    text = text.replace('<url>', ' special_token_url ')
    text = text.replace('<username>', ' special_token_user ')
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s:]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_features_adv_inference(text, vec_word, vec_char):
    tfidf_word = vec_word.transform([text])
    tfidf_char = vec_char.transform([text])
    word_counts = [len(text.split())]
    word_counts_sparse = csr_matrix(word_counts).T
    avg_word_len = [np.mean([len(w) for w in text.split()]) if text.split() else 0]
    avg_word_len_sparse = csr_matrix(avg_word_len).T
    return hstack([tfidf_word, tfidf_char, word_counts_sparse, avg_word_len_sparse])

# Preprocessing cho Deep Learning (BiLSTM, TextCNN)
_DEMOJIZE_COLON_BLOCK = re.compile(r':([-+a-zA-Z0-9][-a-zA-Z0-9_]*):')
def expand_demojized_tokens(text: str) -> str:
    if not text: return ''
    def repl(m):
        inner = m.group(1)
        if inner.isdigit(): return m.group(0)
        phrase = inner.replace('_', ' ').strip()
        return f' {phrase} ' if phrase else m.group(0)
    t = _DEMOJIZE_COLON_BLOCK.sub(repl, text)
    return re.sub(r'\s+', ' ', t).strip()

def normalize_placeholder_tokens(text: str) -> str:
    if not text: return ''
    t = re.sub(r'<username>', ' USERPLACEHOLDER ', text, flags=re.I)
    t = re.sub(r'<url>', ' URLPLACEHOLDER ', t, flags=re.I)
    t = re.sub(r'<name>', ' NAMEPLACEHOLDER ', t, flags=re.I)
    t = re.sub(r'<number>', ' NUMBERPLACEHOLDER ', t, flags=re.I)
    return re.sub(r'\s+', ' ', t).strip()

def tokenize(text: str):
    if not text: return []
    t = normalize_placeholder_tokens(text.strip())
    t = expand_demojized_tokens(t)
    return word_tokenize(t)

def encode_texts(texts, word2idx, max_len=256):
    unk, pad = word2idx.get('<unk>', 1), word2idx.get('<pad>', 0)
    batch_ids, batch_lens = [], []
    for t in texts:
        ids = [word2idx.get(w, unk) for w in tokenize(t)]
        if len(ids) > max_len: ids = ids[:max_len]
        batch_lens.append(len(ids) if ids else 1)
        while len(ids) < max_len: ids.append(pad)
        batch_ids.append(ids)
    return torch.tensor(batch_ids, dtype=torch.long), torch.tensor(batch_lens, dtype=torch.long)

## 2. PYTORCH DEEP LEARNING CLASSES

In [ ]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_out, mask):
        scores = self.proj(torch.tanh(rnn_out)).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (weights * rnn_out).sum(dim=1)

class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, padding_idx, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = AttentionPooling(hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        emb = self.dropout(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = x != self.embedding.padding_idx
        ctx = self.attn(out, mask)
        return self.fc(self.dropout(ctx))

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, padding_idx, dropout=0.4, num_filters=128, filter_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.dropout = nn.Dropout(dropout)
        self.convs = nn.ModuleList(nn.Conv1d(embed_dim, num_filters, kernel_size=k) for k in filter_sizes)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x, lengths=None):
        emb = self.dropout(self.embedding(x)).transpose(1, 2)
        pooled = [F.max_pool1d(F.relu(conv(emb)), kernel_size=emb.size(2)-conv.kernel_size[0]+1).squeeze(2) for conv in self.convs]
        return self.fc(self.dropout(torch.cat(pooled, dim=1)))

## 3. LOAD TẤT CẢ MODEL
**Lưu ý:** Bạn cần phải chỉnh sửa lại các đường dẫn (`BERT_PATH`, `ML_DIR`, `DL_DIR`) sao cho khớp với thư mục Model bạn Upload lên Kaggle.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

ID2LABEL = {0: "Normal", 1: "Depression", 2: "Suicidal", 3: "Anxiety"}
LABEL_ORDER_DL = ["Normal", "Depression", "Suicidal", "Anxiety"]

# --------------------------------------------------------------------
# TODO: THAY ĐỔI ĐƯỜNG DẪN THEO DATASET TRÊN KAGGLE CỦA BẠN
# --------------------------------------------------------------------
BERT_PATH = "/kaggle/input/models/thaidat733/models-cs221/tensorflow2/default/1/BERT-Finetuning"
ML_DIR = "/kaggle/input/YOUR-DATASET-NAME/Models"
DL_DIR = "/kaggle/input/YOUR-DATASET-NAME/dl_outputs"
# --------------------------------------------------------------------

# 3.1 Load BERT
try:
    print("Loading BERT...")
    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)
    bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_PATH).to(device)
    bert_model.eval()
    print("\tBERT Loaded!")
except Exception as e:
    print("\tBERT failed:", e)
    bert_model = None

# 3.2 Load ML Models (SVM, LightGBM)
try:
    print("Loading ML Models...")
    svm_model = joblib.load(f"{ML_DIR}/svm_model.pkl")
    lgb_model = joblib.load(f"{ML_DIR}/lgb_model.pkl")
    tfidf_word = joblib.load(f"{ML_DIR}/tfidf_word.pkl")
    tfidf_char = joblib.load(f"{ML_DIR}/tfidf_char.pkl")
    le = joblib.load(f"{ML_DIR}/label_encoder.pkl")
    print("\tML Models Loaded!")
except Exception as e:
    print("\tML Models failed:", e)
    svm_model, lgb_model, tfidf_word, tfidf_char, le = None, None, None, None, None

# 3.3 Load DL Models (BiLSTM, TextCNN)
try:
    print("Loading DL Models...")
    bilstm_ckpt = torch.load(f"{DL_DIR}/bilstm_final.pt", map_location=device)
    word2idx_bilstm = bilstm_ckpt["word2idx"]
    embed_dim = bilstm_ckpt.get("args", {}).get("embed_dim", 300)
    hidden_dim = bilstm_ckpt.get("args", {}).get("hidden_dim", 128)
    bilstm_model = BiLSTMAttention(len(word2idx_bilstm), embed_dim, hidden_dim, 4, word2idx_bilstm.get("<pad>", 0)).to(device)
    bilstm_model.load_state_dict(bilstm_ckpt["model_state"])
    bilstm_model.eval()
    
    textcnn_ckpt = torch.load(f"{DL_DIR}/textcnn_final.pt", map_location=device)
    word2idx_textcnn = textcnn_ckpt["word2idx"]
    embed_dim = textcnn_ckpt.get("args", {}).get("embed_dim", 300)
    num_filters = textcnn_ckpt.get("args", {}).get("num_filters", 128)
    textcnn_model = TextCNN(len(word2idx_textcnn), embed_dim, 4, word2idx_textcnn.get("<pad>", 0), num_filters=num_filters).to(device)
    textcnn_model.load_state_dict(textcnn_ckpt["model_state"])
    textcnn_model.eval()
    print("\tDL Models Loaded!")
except Exception as e:
    print("\tDL Models failed:", e)
    bilstm_model, textcnn_model = None, None

## 4. INFERENCE FUNCTIONS

In [ ]:
def predict_with_bert(text):
    if not bert_model: return {"label": "Error (BERT not loaded)", "confidence": 0}
    clean = preprocess_text_bert(text)
    inputs = bert_tokenizer(clean, truncation=True, padding="max_length", max_length=300, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
        conf, pred = torch.max(probs, dim=-1)
    return {"label": ID2LABEL[pred.item()], "confidence": conf.item()}

def predict_with_ml(text, model):
    if not model or not tfidf_word: return {"label": "Error (ML not loaded)", "confidence": 0}
    clean = preprocess_text_ml(text)
    features = get_features_adv_inference(clean, tfidf_word, tfidf_char)
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(features)[0]
        pred_idx = np.argmax(probs)
        conf = probs[pred_idx]
    else:
        pred_idx = model.predict(features)[0]
        dec = model.decision_function(features)[0]
        exp = np.exp(dec - np.max(dec))
        probs = exp / exp.sum()
        conf = probs[pred_idx]
    return {"label": le.inverse_transform([pred_idx])[0], "confidence": float(conf)}

def predict_with_dl(text, model, word2idx):
    if not model or not word2idx: return {"label": "Error (DL not loaded)", "confidence": 0}
    x, lengths = encode_texts([text], word2idx, max_len=256)
    with torch.no_grad():
        logits = model(x.to(device), lengths.to(device))
        probs = torch.softmax(logits, dim=-1)
        conf, pred = torch.max(probs, dim=-1)
    return {"label": LABEL_ORDER_DL[pred.item()], "confidence": conf.item()}

## 5. FASTAPI & NGROK SETUP

In [ ]:
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class PredictRequest(BaseModel):
    text: str
    model: str = "BERT"

@app.post("/api/predict")
def predict(req: PredictRequest):
    if req.model == "BERT":
        return predict_with_bert(req.text)
    elif req.model == "BiLSTM":
        return predict_with_dl(req.text, bilstm_model, word2idx_bilstm)
    elif req.model == "TextCNN":
        return predict_with_dl(req.text, textcnn_model, word2idx_textcnn)
    elif req.model == "SVM":
        return predict_with_ml(req.text, svm_model)
    elif req.model == "LightGBM":
        return predict_with_ml(req.text, lgb_model)
    return {"label": "Unknown Model", "confidence": 0.0}

from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("ngrok")
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
    public_url = ngrok.connect(8000).public_url
    print("="*60)
    print("API IS RUNNING (MULTI-MODEL SUPPORT)")
    print(f"ENDPOINT: {public_url}/api/predict")
    print("="*60)
except Exception as e:
    print("Ngrok Error:", e)

nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
server.run()